# 04 · LoRA SFT and DPO (methods 4 and 5)

Trains on the synthetic data (Wei et al. style, half of the examples have the user right), never on the
benchmarks. Training templates are different from evaluation templates. DPO uses the same network with the adapter
switched off as the reference model, so the first loss should be about ln 2 ≈ 0.693.

## Setup

Runtime → Change runtime type → **T4 GPU**. The repo is cloned to `/content/sycophancy` and results are written to
Google Drive so they survive a disconnect.

In [ ]:
import os
if not os.path.exists("/content/sycophancy"):
    !git clone https://github.com/namita-tj/sycophancy /content/sycophancy
%cd /content/sycophancy
!pip install -q -e ".[all]"

In [ ]:
# Gated models (Llama 3.2, Gemma 2) need a Hugging Face token with access accepted on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Keep results on Drive.
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sycophancy/results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s $RESULTS results

In [ ]:
# Evaluation data (MMLU + Perez et al.) and synthetic training data. Only needed once per runtime.
if not os.path.exists("data/eval/pressure.jsonl"):
    !bash scripts/prepare_data.sh
!wc -l data/eval/*.jsonl data/train/*.jsonl

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TAG = MODEL.split("/")[-1].lower()
DTYPE = "fp16"   # a T4 has no bf16; use bf16 on an A100 or L4

In [ ]:
!syco train sft --model {MODEL} --dtype {DTYPE} --train data/train/synthetic.jsonl --epochs 1 --batch-size 8 --out adapters/{TAG}-sft

In [ ]:
!syco train dpo --model {MODEL} --dtype {DTYPE} --train data/train/synthetic.jsonl --epochs 1 --batch-size 4 --beta 0.1 --out adapters/{TAG}-dpo

In [ ]:
import json
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for i, m in enumerate(["sft", "dpo"]):
    log = json.load(open(f"adapters/{TAG}-{m}/train_log.json"))
    ax[i].plot(log["losses"]); ax[i].set_title(m.upper()); ax[i].set_xlabel("step")
print("DPO first loss:", json.load(open(f"adapters/{TAG}-dpo/train_log.json"))["losses"][0])

In [ ]:
for m in ["sft", "dpo"]:
    !syco eval --model {MODEL} --adapter adapters/{TAG}-{m} --dtype {DTYPE} --data data/eval --split test --out results/{TAG}-{m}
!syco compare results/{TAG}-baseline results/{TAG}-sft results/{TAG}-dpo --out results/{TAG}-finetune.md
from IPython.display import Markdown
Markdown(open(f"results/{TAG}-finetune.md").read())

In [ ]:
# Copy the adapters to Drive (they're small).
!mkdir -p /content/drive/MyDrive/sycophancy/adapters && cp -r adapters/{TAG}-* /content/drive/MyDrive/sycophancy/adapters/